Data preprocessing -resizing

In [ ]:
import os
import cv2
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt

def preprocess_images(input_folder, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    for filename in os.listdir(input_folder):
        img_path = os.path.join(input_folder, filename)
        img = cv2.imread(img_path)
        if img is not None:
            h, w, _ = img.shape
            scale = 640 / max(h, w)
            new_h, new_w = int(h * scale), int(w * scale)
            resized_img = cv2.resize(img, (new_w, new_h))

            # Create a new 640x640 image and place the resized image in the center
            result_img = cv2.copyMakeBorder(
                resized_img,
                top=(640 - new_h) // 2,
                bottom=(640 - new_h + 1) // 2,
                left=(640 - new_w) // 2,
                right=(640 - new_w + 1) // 2,
                borderType=cv2.BORDER_CONSTANT,
                value=[0, 0, 0]  # Black border
            )

            output_path = os.path.join(output_folder, filename)
            cv2.imwrite(output_path, result_img)
        else:
            print(f"Failed to read image {img_path}")

def convert_voc_to_yolo(voc_folder, yolo_folder, image_folder):
    if not os.path.exists(yolo_folder):
        os.makedirs(yolo_folder)

    for xml_file in os.listdir(voc_folder):
        if not xml_file.endswith('.xml'):
            continue

        xml_path = os.path.join(voc_folder, xml_file)
        tree = ET.parse(xml_path)
        root = tree.getroot()

        image_file = root.find('filename').text
        image_path = os.path.join(image_folder, image_file)
        img = cv2.imread(image_path)
        if img is None:
            print(f"Image {image_file} not found, skipping annotation conversion.")
            continue
        h, w, _ = img.shape

        yolo_annotation = []
        for obj in root.findall('object'):
            cls = obj.find('name').text
            cls_id = classes.index(cls)  # Assumes 'classes' is a predefined list of class names
            xmlbox = obj.find('bndbox')
            b = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text),
                 float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text))
            bb = convert_bbox((w, h), b)
            yolo_annotation.append(f"{cls_id} " + " ".join([str(a) for a in bb]))

        yolo_path = os.path.join(yolo_folder, xml_file.replace('.xml', '.txt'))
        with open(yolo_path, 'w') as f:
            f.write("\n".join(yolo_annotation))

def convert_bbox(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    x = x * dw
    w = w * dw
    y = y * dh
    h = h * dh
    return (x, y, w, h)

# Define paths based on your VS Code directory
dataset_path = './data'
output_base_path = './data_preprocessed'

subfolders = ['Rain', 'Sand', 'Fog', 'Snow']
classes = ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'train']  # Define your classes here

for folder in subfolders:
    input_folder = os.path.join(dataset_path, folder)
    yolo_folder = os.path.join(input_folder, folder + '_YOLO_darknet')
    voc_folder = os.path.join(input_folder, folder + '_PASCAL_VOC')
    output_folder = os.path.join(output_base_path, folder)

    # Preprocess images
    preprocess_images(input_folder, output_folder)

    # Convert PASCAL VOC annotations to YOLO format
    convert_voc_to_yolo(voc_folder, yolo_folder, input_folder)

# Example usage to display images
def display_images(original_folder, preprocessed_folder, image_filename):
    original_img_path = os.path.join(original_folder, image_filename)
    preprocessed_img_path = os.path.join(preprocessed_folder, image_filename)

    # Check if the images exist
    if not os.path.exists(original_img_path):
        print(f"Original image not found: {original_img_path}")
        return
    if not os.path.exists(preprocessed_img_path):
        print(f"Preprocessed image not found: {preprocessed_img_path}")
        return

    original_img = cv2.imread(original_img_path)
    preprocessed_img = cv2.imread(preprocessed_img_path)

    if original_img is None:
        print(f"Failed to read original image {original_img_path}")
        return
    if preprocessed_img is None:
        print(f"Failed to read preprocessed image {preprocessed_img_path}")
        return

    # Convert images from BGR to RGB
    original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    preprocessed_img = cv2.cvtColor(preprocessed_img, cv2.COLOR_BGR2RGB)

    # Display images
    plt.figure(figsize=(10, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(original_img)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(preprocessed_img)
    plt.title('Preprocessed Image')
    plt.axis('off')

    plt.show()

# Display example image
folder = 'Sand'
original_folder = os.path.join(dataset_path, folder)
preprocessed_folder = os.path.join(output_base_path, folder)
image_filename = os.listdir(original_folder)[2]  # Display an image from the folder

display_images(original_folder, preprocessed_folder, image_filename)


Training using YOLOv11

In [ ]:
import os
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO  # Import YOLOv11

# Define the class names (YOLOv11 defaults to COCO classes, adjust if needed)
class_names = ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'train']

def load_model_and_weights(variant):
    # Load the YOLOv11 model
    model = YOLO(f"yolo11{variant}.pt")  # Replace with path to your YOLOv11 weights
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)  # Move the model to the appropriate device
    return model

def infer_image(model, image_path, conf_threshold=0.3, iou_threshold=0.65):
    # Load and preprocess the image
    origin_img = cv2.imread(image_path)
    if origin_img is None:
        print(f"Failed to read image {image_path}")
        return None

    # Use YOLOv11's predict method
    results = model.predict(origin_img, conf=conf_threshold, iou=iou_threshold)

    if len(results) > 0 and results[0].boxes.xyxy is not None:
        bboxes = results[0].boxes.xyxy
        scores = results[0].boxes.conf
        cls_ids = results[0].boxes.cls.int()

        # Visualize the results with bounding boxes
        result_image = results[0].plot()  # YOLOv11 specific visualization

        # Print detected class names and scores
        for bbox, score, cls_id in zip(bboxes, scores, cls_ids):
            if cls_id < len(class_names):  # Check if the class ID is within the class_names list
                cls_name = class_names[cls_id]
                print(f"Detected {cls_name} with score {score}")
            else:
                print(f"Class ID {cls_id} is not in the list of class names, skipping...")
    else:
        result_image = origin_img

    return result_image

# Paths to dataset and output
dataset_path = './data_preprocessed'
output_base_path = './data_results'

subfolders = ['Rain', 'Sand', 'Fog', 'Snow']  # Adjust to your subfolder structure
variants = ['n', 's', 'm', 'l']  # YOLOv11 variants (nano, small, medium, large)

# Run inference and save results for each variant
for variant in variants:
    model = load_model_and_weights(variant)  # Load YOLOv11 model for each variant

    for subfolder in subfolders:
        folder_path = os.path.join(dataset_path, subfolder)
        output_folder = os.path.join(output_base_path, f"{subfolder}_yolo11_{variant}")
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)

        for filename in os.listdir(folder_path):
            img_path = os.path.join(folder_path, filename)
            result = infer_image(model, img_path, conf_threshold=0.5)
            if result is not None:
                # Convert BGR to RGB for displaying
                plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
                plt.title(f'Detected Results - {subfolder} - YOLOv11 {variant}')
                plt.axis('off')  # Hide axes
                plt.show()

                # Save the result image
                output_img_path = os.path.join(output_folder, filename)
                cv2.imwrite(output_img_path, result)
            else:
                print(f"No results for {filename}")



Image enhancement (unsharp masking)

In [ ]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt

# Function to show images before and after enhancement
def display_images(original, enhanced, title):
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.title('Original Image')
    plt.imshow(original, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title(title)
    plt.imshow(enhanced, cmap='gray')
    plt.axis('off')

    plt.show()

# Folder paths
base_input_folder = './data'  # Base path to your class folders (e.g., 'data/fog', 'data/rain', etc.)
output_base_folder = 'data_enhanced/unsharp'  # Base output folder for enhanced images

# List of class subfolders
subfolders = ['fog', 'rain', 'sand', 'snow']  # Adjust as needed

# Loop through each class subfolder
for subfolder in subfolders:
    input_folder = os.path.join(base_input_folder, subfolder)
    output_folder = os.path.join(output_base_folder, subfolder)
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.endswith('.jpg') or filename.endswith('.png'):
            img_path = os.path.join(input_folder, filename)
            original = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            # Unsharp Masking
            gaussian_blur = cv2.GaussianBlur(original, (5, 5), 1.5)
            unsharp_masked = cv2.addWeighted(original, 1.5, gaussian_blur, -0.5, 0)

            # Save enhanced image
            cv2.imwrite(os.path.join(output_folder, filename), unsharp_masked)

            # Display images (optional)
           # display_images(original, unsharp_masked, f'Unsharp Masked Image - {subfolder}')


Training YOLOv11 + Image enhancement

In [ ]:
import os
import torch
import cv2
import numpy as np
from ultralytics import YOLO  # Import YOLOv11

# Define the class names (YOLOv11 defaults to COCO classes, adjust if needed)
class_names = ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'train']

def load_model_and_weights(weight_path):
    # Load the YOLOv11 model
    model = YOLO(weight_path)  # Specify the full path to your YOLOv11 weights
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)  # Move the model to the appropriate device
    return model

def infer_image(model, image_path, conf_threshold=0.3, iou_threshold=0.65):
    # Use YOLOv11's predict method directly on the image path
    results = model.predict(source=image_path, conf=conf_threshold, iou=iou_threshold)

    # Check if there are any detections
    if results:
        # Visualize the results with bounding boxes
        result_image = results[0].plot()  # YOLOv11 specific visualization
        detections = results[0].boxes

        # Print detected class names and scores
        for box in detections:
            x1, y1, x2, y2 = box.xyxy[0].tolist()  # Get the bounding box coordinates as a list
            score = box.conf[0].item()  # Get the confidence score
            cls_id = int(box.cls[0])  # Get the class ID
            if cls_id < len(class_names):  # Check if the class ID is within the class_names list
                cls_name = class_names[cls_id]
                print(f"Detected {cls_name} with score {score} at [{x1}, {y1}, {x2}, {y2}]")
            else:
                print(f"Class ID {cls_id} is not in the list of class names, skipping...")

        return result_image
    else:
        print(f"No detections for image {image_path}")
        return cv2.imread(image_path)  # Return the original image if no detections

# Paths to enhanced dataset and output
enhanced_dataset_path = './data_enhanced/unsharp'
output_base_path = './data_results_enhanced'

subfolders = ['Rain', 'Sand', 'Fog', 'Snow']  # Adjust to your subfolder structure if necessary

# Define paths for each YOLOv11 variant weight
weights_paths = {
    'n': "path/to/yolo11n.pt",
    's': "path/to/yolo11s.pt",
    'm': "path/to/yolo11m.pt",
    'l': "path/to/yolo11l.pt"
}

# Run inference and save results for each variant
for variant, weight_path in weights_paths.items():
    model = load_model_and_weights(weight_path)  # Load YOLOv11 model for each variant

    for subfolder in subfolders:
        folder_path = os.path.join(enhanced_dataset_path, subfolder)
        output_folder = os.path.join(output_base_path, f"{subfolder}_yolo11_{variant}")
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)

        for filename in os.listdir(folder_path):
            img_path = os.path.join(folder_path, filename)
            result = infer_image(model, img_path, conf_threshold=0.5)
            if result is not None:
                # Save the result image
                output_img_path = os.path.join(output_folder, filename)
                cv2.imwrite(output_img_path, result)
            else:
                print(f"No results for {filename}")


OSError: [WinError 127] The specified procedure could not be found. Error loading "c:\Users\Shoro\anaconda3\envs\project\Lib\site-packages\torch\lib\torch_cuda.dll" or one of its dependencies.

Results


In [ ]:
import os
import cv2
import torch
from ultralytics import YOLO  # Ensure correct YOLO import

# Define paths to YOLOv11 weights
yolo11_weights = {
    'n': r"C:\Users\Shoro\Desktop\proj\path\to\yolo11n.pt",
    's': r"C:\Users\Shoro\Desktop\proj\path\to\yolo11s.pt",
    'm': r"C:\Users\Shoro\Desktop\proj\path\to\yolo11m.pt",
    'l': r"C:\Users\Shoro\Desktop\proj\path\to\yolo11l.pt",
}

# Enhanced dataset path
enhanced_dataset_base = r"C:\Users\Shoro\Desktop\proj\data_results_enhanced"
categories = ['Fog', 'Sand', 'Snow', 'Rain']

# Output folder names (YOLO format)
output_suffix = "_YOLO_darknet"

# Function to prepare output folders
def prepare_output_folder(category_path):
    output_folder = os.path.join(category_path, f"{category_path.split(os.sep)[-1]}{output_suffix}")
    os.makedirs(output_folder, exist_ok=True)
    return output_folder

# Load YOLOv11 model (Ensure the model loading is independent of dataset YAML file)
def load_yolo11_model(weights_path):
    try:
        model = YOLO(weights_path)  # Directly use the correct path here
        model.eval()  # Set the model to evaluation mode
        return model
    except Exception as e:
        print(f"Error loading model: {e}")
        return None

# Process each YOLO version
for yolo_version, weight_path in yolo11_weights.items():
    if not os.path.exists(weight_path):
        print(f"Error: Weight file not found - {weight_path}")
        continue

    model = load_yolo11_model(weight_path)
    if model is None:
        continue

    for category in categories:
        category_path = os.path.join(enhanced_dataset_base, category)
        output_folder = prepare_output_folder(category_path)

        for image_file in os.listdir(category_path):
            if image_file.endswith(('.jpg', '.png')):
                image_path = os.path.join(category_path, image_file)
                txt_output_path = os.path.join(output_folder, image_file.replace('.jpg', '.txt').replace('.png', '.txt'))

                # Read image
                image = cv2.imread(image_path)
                if image is None:
                    print(f"Error reading image: {image_path}")
                    continue

                # Run inference
                results = model(image)

                # Extract detections (results.xywh should contain the detections)
                detections = results.xywh[0].cpu().numpy()

                # Count the number of detections
                detection_count = len(detections)

                # Print the count for debugging
                print(f"Number of detections for {image_file}: {detection_count}")

                # Save the count to a text file (optional)
                count_txt_path = os.path.join(output_folder, image_file.replace('.jpg', '_count.txt').replace('.png', '_count.txt'))
                with open(count_txt_path, 'w') as count_file:
                    count_file.write(f"Number of detections: {detection_count}\n")

                # Write the detections in YOLO format to the output text file
                with open(txt_output_path, 'w') as txt_file:
                    for detection in detections:
                        class_id = int(detection[5])
                        x_center, y_center, width, height = detection[:4]
                        txt_file.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

        print(f"Processed category '{category}' for YOLOv11-{yolo_version}, results saved in {output_folder}")

print("YOLOv11 inference completed for all enhanced images.")


New https://pypi.org/project/ultralytics/8.3.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.27  Python-3.12.5 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: task=detect, mode=train, model=C:\Users\Shoro\Desktop\proj\path\to\yolo11n.pt, data=/usr/src/ultralytics/ultralytics/cfg/datasets/coco.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train7, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, a

In [ ]:
# Process for data_results
dataset_path = data_results_path  # Path to original dataset

for subfolder in subfolders:
    # Dynamically create the folder path for each subfolder (Fog, Rain, etc.)
    folder_path = os.path.join(dataset_path, f"{subfolder}_yolo11_l")  # Specific path for Fog_yolo11_l, Rain_yolo11_l, etc.

    # Ensure the folder exists before proceeding
    if not os.path.exists(folder_path):
        print(f"Skipping non-existent folder: {folder_path}")
        continue

    # Define the output folder for YOLOv11 model (for each class and model version)
    class_output_folder = os.path.join(folder_path, "YOLO_darknet")  # This will be inside Fog_yolo11_l, Rain_yolo11_l, etc.

    # Create the output folder if it doesn't exist
    if not os.path.exists(class_output_folder):
        os.makedirs(class_output_folder)

    # Iterate through all YOLOv11 versions (s, m, l)
    for version, weight_path in yolo11_weights.items():
        # Load the model for the current YOLOv11 version
        model = load_model_and_weights(weight_path)
        print(f"Processing with YOLOv11-{version.upper()} model...")

        # Iterate through images in the folder
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):  # Adjusted to handle multiple image formats
                img_path = os.path.join(folder_path, filename)
                infer_and_save_results(model, img_path, class_output_folder, conf_threshold=0.5)
            else:
                print(f"Skipping non-image file: {filename}")

# Process for data_results_enhanced
dataset_path = enhanced_dataset_path  # Path to enhanced dataset

for subfolder in subfolders:
    # Dynamically create the folder path for each subfolder (Fog, Rain, etc.)
    folder_path = os.path.join(dataset_path, f"{subfolder}_yolo11_l")  # Specific path for Fog_yolo11_l, Rain_yolo11_l, etc.

    # Ensure the folder exists before proceeding
    if not os.path.exists(folder_path):
        print(f"Skipping non-existent folder: {folder_path}")
        continue

    # Define the output folder for YOLOv11 model (for each class and model version)
    class_output_folder = os.path.join(folder_path, "YOLO_darknet")  # This will be inside Fog_yolo11_l, Rain_yolo11_l, etc.

    # Create the output folder if it doesn't exist
    if not os.path.exists(class_output_folder):
        os.makedirs(class_output_folder)

    # Iterate through all YOLOv11 versions (s, m, l)
    for version, weight_path in yolo11_weights.items():
        # Load the model for the current YOLOv11 version
        model = load_model_and_weights(weight_path)
        print(f"Processing with YOLOv11-{version.upper()} model...")

        # Iterate through images in the folder
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):  # Adjusted to handle multiple image formats
                img_path = os.path.join(folder_path, filename)
                infer_and_save_results(model, img_path, class_output_folder, conf_threshold=0.5)
            else:
                print(f"Skipping non-image file: {filename}")


Processing with YOLOv11-L model...

image 1/1 C:\Users\Shoro\Desktop\proj\data_results\Fog_yolo11_l\foggy-001.jpg: 640x640 1 person, 1 car, 23.4ms
Speed: 4.0ms preprocess, 23.4ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 C:\Users\Shoro\Desktop\proj\data_results\Fog_yolo11_l\foggy-002.jpg: 640x640 (no detections), 33.3ms
Speed: 0.0ms preprocess, 33.3ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 C:\Users\Shoro\Desktop\proj\data_results\Fog_yolo11_l\foggy-003.jpg: 640x640 2 cars, 1 truck, 18.6ms
Speed: 0.0ms preprocess, 18.6ms inference, 12.9ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 C:\Users\Shoro\Desktop\proj\data_results\Fog_yolo11_l\foggy-004.jpg: 640x640 2 cars, 1 bus, 30.1ms
Speed: 0.0ms preprocess, 30.1ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 C:\Users\Shoro\Desktop\proj\data_results\Fog_yolo11_l\foggy-006.jpg: 640x640 (no detections), 31.4ms
Speed: 0.0ms prepr

In [ ]:
import os
from prettytable import PrettyTable

# Define the class names based on your dataset
class_names = ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'train']

# Paths to results
results_path = r"C:\Users\Shoro\Desktop\proj\data_results"
results_path_enhanced = r"C:\Users\Shoro\Desktop\proj\data_results_enhanced"

# Subfolders for categories
subfolders = ['Fog', 'Sand', 'Snow', 'Rain']

# Function to count objects in YOLO text files
def count_objects_in_txt(folder_path):
    counts = {cls: 0 for cls in class_names}  # Initialize counts
    if not os.path.exists(folder_path):
        print(f"Warning: Folder does not exist - {folder_path}")
        return counts

    # Loop through all .txt files in the given folder
    for filename in os.listdir(folder_path):
        if filename.endswith('.txt'):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'r') as f:
                lines = f.readlines()  # Read all lines in the file
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) > 0:
                        class_id = int(parts[0])  # Class ID is the first element
                        if class_id < len(class_names):
                            class_name = class_names[class_id]  # Get class name
                            counts[class_name] += 1  # Increment the count for that class
    return counts

# Generate tables for each category
for category in subfolders:
    table = PrettyTable()
    table.field_names = ["Class", "Detected (Original)", "Detected (Enhanced)"]

    # Define folder paths for original and enhanced results
    original_folder = os.path.join(results_path, f"{category}_yolo11_l", "YOLO_darknet")
    enhanced_folder = os.path.join(results_path_enhanced, f"{category}_yolo11_l", "YOLO_darknet")

    # Check if the directories exist for both original and enhanced
    if not os.path.exists(original_folder):
        print(f"Warning: Original folder does not exist - {original_folder}")
    if not os.path.exists(enhanced_folder):
        print(f"Warning: Enhanced folder does not exist - {enhanced_folder}")

    # Count objects for original and enhanced
    original_counts = count_objects_in_txt(original_folder)
    enhanced_counts = count_objects_in_txt(enhanced_folder)

    # Add data to the table
    for class_name in class_names:
        table.add_row([class_name, original_counts[class_name], enhanced_counts[class_name]])

    # Print the table for the category
    print(f"\nCategory: {category}")
    print(table)



Category: Fog
+------------+---------------------+---------------------+
|   Class    | Detected (Original) | Detected (Enhanced) |
+------------+---------------------+---------------------+
|   person   |          18         |          20         |
|  bicycle   |          0          |          0          |
|    car     |         390         |         357         |
| motorcycle |          1          |          2          |
|    bus     |          0          |          0          |
|   truck    |          10         |          5          |
|   train    |          0          |          0          |
+------------+---------------------+---------------------+

Category: Sand
+------------+---------------------+---------------------+
|   Class    | Detected (Original) | Detected (Enhanced) |
+------------+---------------------+---------------------+
|   person   |          26         |          29         |
|  bicycle   |          1          |          2          |
|    car     |         40

In [ ]:
import os
import numpy as np
from sklearn.metrics import auc

def load_labels(label_path):
    """Load YOLO labels from the given path."""
    labels = {}
    for file in os.listdir(label_path):
        if file.endswith(".txt"):
            with open(os.path.join(label_path, file), 'r') as f:
                boxes = []
                for line in f.readlines():
                    parts = line.strip().split()
                    class_id = int(parts[0])
                    bbox = list(map(float, parts[1:]))
                    boxes.append([class_id, *bbox])
                labels[file] = boxes
    return labels

def compute_iou(box1, box2):
    """Calculate IoU for two bounding boxes in YOLO format."""
    x1_min, y1_min = box1[1] - box1[3]/2, box1[2] - box1[4]/2
    x1_max, y1_max = box1[1] + box1[3]/2, box1[2] + box1[4]/2
    x2_min, y2_min = box2[1] - box2[3]/2, box2[2] - box2[4]/2
    x2_max, y2_max = box2[1] + box2[3]/2, box2[2] + box2[4]/2

    inter_w = max(0, min(x1_max, x2_max) - max(x1_min, x2_min))
    inter_h = max(0, min(y1_max, y2_max) - max(y1_min, y2_min))
    intersection = inter_w * inter_h

    area1 = (x1_max - x1_min) * (y1_max - y1_min)
    area2 = (x2_max - x2_min) * (y2_max - y2_min)
    union = area1 + area2 - intersection

    return intersection / union

def calculate_ap(gt_boxes, pred_boxes, iou_threshold=0.5):
    """Calculate Average Precision (AP) for a single class."""
    if len(gt_boxes) == 0 or len(pred_boxes) == 0:
        return 0.0

    tp = []
    fp = []
    confs = []
    gt_matched = set()

    for pred in pred_boxes:
        pred_class, x, y, w, h, confidence = pred
        confs.append(confidence)
        matched = False

        for i, gt in enumerate(gt_boxes):
            gt_class, gx, gy, gw, gh = gt
            if gt_class != pred_class or i in gt_matched:
                continue

            iou = compute_iou(pred, gt)
            if iou >= iou_threshold:
                matched = True
                gt_matched.add(i)
                tp.append(1)
                fp.append(0)
                break

        if not matched:
            tp.append(0)
            fp.append(1)

    if len(tp) == 0 or len(fp) == 0:
        return 0.0

    tp = np.cumsum(tp)
    fp = np.cumsum(fp)
    confs = np.array(confs)
    indices = np.argsort(-confs)

    tp = tp[indices]
    fp = fp[indices]

    precisions = tp / (tp + fp + 1e-16)
    recalls = tp / len(gt_boxes)

    if len(recalls) == 0 or len(precisions) == 0:
        return 0.0

    return auc(recalls, precisions)


def calculate_map(gt_path, pred_path):
    """Calculate mean Average Precision (mAP) for all classes."""
    gt_labels = load_labels(gt_path)
    pred_labels = load_labels(pred_path)

    aps = []
    for file, gt_boxes in gt_labels.items():
        pred_boxes = pred_labels.get(file, [])
        ap = calculate_ap(gt_boxes, pred_boxes)
        aps.append(ap)

    return np.mean(aps)

# Paths to your data
gt_path = "C:\\Users\\Shoro\\Desktop\\proj\\data"
original_pred_path = "C:\\Users\\Shoro\\Desktop\\proj\\data_results"
enhanced_pred_path = "C:\\Users\\Shoro\\Desktop\\proj\\data_results_enhanced"

# Calculate mAP
original_map = calculate_map(gt_path, original_pred_path)
enhanced_map = calculate_map(gt_path, enhanced_pred_path)

print(f"mAP for original dataset: {original_map:.4f}")
print(f"mAP for enhanced dataset: {enhanced_map:.4f}")


mAP for original dataset: nan
mAP for enhanced dataset: nan


c:\Users\Shoro\anaconda3\envs\project\Lib\site-packages\numpy\_core\fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Shoro\anaconda3\envs\project\Lib\site-packages\numpy\_core\_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [ ]:
import os
import torch
import cv2
import numpy as np
from yolox.data.data_augment import ValTransform
from yolox.utils import postprocess, vis
from yolox.exp import get_exp

# Define the class names
class_names = ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'train']

def load_model_and_weights(variant):
    # Load the experiment configuration for YOLOX
    exp = get_exp(None, f"yolox-{variant}")
    model = exp.get_model()
    model.eval()

    # Load the model weights
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Specify the full path to the model weight files
    ckpt = torch.load(f"C:/Users/Shoro/YOLOX/yolox_{variant}.pth", map_location=device)

    model.load_state_dict(ckpt["model"])
    model.to(device)  # Move the model to the appropriate device

    return model, exp

def infer_image(model, exp, image_path, conf_threshold=0.3, nms_threshold=0.65):
    # Load and preprocess the image
    origin_img = cv2.imread(image_path)
    if origin_img is None:
        print(f"Failed to read image {image_path}")
        return None

    val_transform = ValTransform()
    img, _ = val_transform(origin_img, None, exp.test_size)
    img = torch.from_numpy(img).unsqueeze(0)

    # Move image tensor to the appropriate device
    device = next(model.parameters()).device  # Get the device of the model
    img = img.to(device)  # Move the image tensor to the same device

    with torch.no_grad():
        # Perform inference
        outputs = model(img)
        outputs = postprocess(outputs, num_classes=len(class_names), conf_thre=conf_threshold, nms_thre=nms_threshold)

    if outputs[0] is not None:
        bboxes = outputs[0][:, 0:4]
        scores = outputs[0][:, 4] * outputs[0][:, 5]
        cls_ids = outputs[0][:, 6].int()

        # Debugging: print class IDs and scores
        print(f"Class IDs: {cls_ids}")
        print(f"Scores: {scores}")

        for bbox, score, cls_id in zip(bboxes, scores, cls_ids):
            cls_name = class_names[cls_id]
            print(f"Detected {cls_name} with score {score}")

        # Visualize the results (but we won't show it)
        result_image = vis(origin_img, bboxes, scores, cls_ids, conf=conf_threshold, class_names=class_names)
    else:
        result_image = origin_img

    return result_image


# Paths to dataset and output
dataset_path = './data_preprocessed'
output_base_path = './data_results'

subfolders = ['Rain', 'Sand', 'Fog', 'Snow']  # Assuming these match the folder names inside 'data'
variants = ['s', 'm', 'l']

# Run inference and save results for each variant
for variant in variants:
    model, exp = load_model_and_weights(variant)  # Load model for each variant

    for subfolder in subfolders:
        folder_path = os.path.join(dataset_path, subfolder)
        output_folder = os.path.join(output_base_path, f"{subfolder}_yolox_{variant}")
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)

        for filename in os.listdir(folder_path):
            img_path = os.path.join(folder_path, filename)
            result = infer_image(model, exp, img_path, conf_threshold=0.5)
            if result is not None:
                # Save the result image
                output_img_path = os.path.join(output_folder, filename)
                cv2.imwrite(output_img_path, result)
            else:
                print(f"No results for {filename}")


In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from yolox.data.data_augment import ValTransform
from yolox.utils import postprocess
from yolox.exp import get_exp

# Define the class names
class_names = ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'train']

# Image Enhancement Function (CLAHE)
def apply_clahe(image):
    img_yuv = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    img_yuv[:, :, 0] = clahe.apply(img_yuv[:, :, 0])
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

# Load the YOLOX Model and Weights
def load_model_and_weights(variant):
    exp = get_exp(None, f"yolox-{variant}")
    model = exp.get_model()
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    ckpt = torch.load(f"C:/Users/Shoro/YOLOX/yolox_{variant}.pth", map_location=device)
    model.load_state_dict(ckpt["model"])
    model.to(device)
    return model, exp

# Inference Function
def infer_image(model, exp, image, conf_threshold=0.3, nms_threshold=0.65):
    val_transform = ValTransform()
    img, _ = val_transform(image, None, exp.test_size)
    img = torch.from_numpy(img).unsqueeze(0)

    device = next(model.parameters()).device
    img = img.to(device)

    with torch.no_grad():
        outputs = model(img)
        outputs = postprocess(outputs, num_classes=len(class_names), conf_thre=conf_threshold, nms_thre=nms_threshold)

    return outputs[0] if outputs[0] is not None else None

# Format Predictions
def format_predictions(outputs):
    bboxes = outputs[:, 0:4].cpu().numpy()  # (x1, y1, x2, y2)
    scores = (outputs[:, 4] * outputs[:, 5]).cpu().numpy()  # conf * class_score
    cls_ids = outputs[:, 6].int().cpu().numpy()
    return bboxes, scores, cls_ids

# Calculate PSNR
def calculate_psnr(original, enhanced):
    mse = np.mean((original.astype(np.float32) - enhanced.astype(np.float32)) ** 2)
    if mse == 0:
        return float('inf')  # If MSE is 0, PSNR is infinite
    max_pixel = 255.0
    psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
    return psnr

# Function to display images
def display_image(image, title):
    plt.figure(figsize=(10, 5))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

# Main Processing Logic
dataset_path = './data_preprocessed'
output_base_path = './data_enhanced'
variant = 's'  # Choose the variant you want to use
subfolder = 'Fog'  # Focus on the fog class

# Initialize results storage
results = {'Technique': [], 'Variant': [], 'PSNR': []}

# Load model for the chosen variant
model, exp = load_model_and_weights(variant)

# Process the Fog subfolder for CLAHE enhancement
folder_path = os.path.join(dataset_path, subfolder)

# Apply CLAHE and calculate PSNR for each image
enhancement_name = 'CLAHE'
for filename in os.listdir(folder_path):
    img_path = os.path.join(folder_path, filename)
    image = cv2.imread(img_path)

    # Apply CLAHE enhancement
    enhanced_image = apply_clahe(image)

    # Display the original and enhanced images
    display_image(image, f'Original Image - {filename}')
    display_image(enhanced_image, f'Enhanced Image - {enhancement_name} - {filename}')

    # Calculate PSNR
    psnr_value = calculate_psnr(image, enhanced_image)

    # Store results
    results['Technique'].append(enhancement_name)
    results['Variant'].append(variant)
    results['PSNR'].append(psnr_value)

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Display and save results
print(results_df)
results_df.to_csv('enhancement_psnr_results.csv', index=False)


In [ ]:
import cv2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def calculate_psnr(original, enhanced):
    mse = np.mean((original - enhanced) ** 2)
    if mse == 0:
        return float('inf')
    psnr = 20 * np.log10(255.0 / np.sqrt(mse))
    return psnr

# Folder path to images
input_folder = 'data/fog'  # Change to your fog class images path
output_folder = 'data_enhanced/hist_eq'  # Output folder for enhanced images

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

results = []

for filename in os.listdir(input_folder):
    if filename.endswith('.jpg') or filename.endswith('.png'):
        img_path = os.path.join(input_folder, filename)
        original = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Histogram Equalization
        enhanced = cv2.equalizeHist(original)
        psnr = calculate_psnr(original, enhanced)

        # Save enhanced image
        cv2.imwrite(os.path.join(output_folder, filename), enhanced)

        # Store results
        results.append({
            'Technique': 'Histogram Equalization',
            'Variant': filename,
            'PSNR': psnr
        })

# Create a DataFrame and save to CSV
results_df = pd.DataFrame(results)
results_df.to_csv('hist_eq_results.csv', index=False)

# Plotting results
plt.figure(figsize=(12, 6))
plt.bar([r['Variant'] for r in results], [r['PSNR'] for r in results])
plt.title('PSNR Results for Histogram Equalization')
plt.xlabel('Image Variants')
plt.ylabel('PSNR')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('hist_eq_psnr_results.png')
plt.show()


In [ ]:
import cv2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Function to calculate PSNR
def calculate_psnr(original, enhanced):
    mse = np.mean((original - enhanced) ** 2)
    if mse == 0:
        return float('inf')
    psnr = 20 * np.log10(255.0 / np.sqrt(mse))
    return psnr

# Folder path to images
input_folder = 'data/fog'  # Change to your fog class images path
output_folder = 'data_enhanced/gaussian_blur'  # Output folder for enhanced images

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

results = []

for filename in os.listdir(input_folder):
    if filename.endswith('.jpg') or filename.endswith('.png'):
        img_path = os.path.join(input_folder, filename)
        original = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Gaussian Blur
        enhanced = cv2.GaussianBlur(original, (5, 5), 0)
        psnr_value = calculate_psnr(original, enhanced)

        # Save enhanced image
        cv2.imwrite(os.path.join(output_folder, filename), enhanced)

        # Store results
        results.append({
            'Technique': 'Gaussian Blur',
            'Variant': filename,
            'PSNR': psnr_value
        })

# Create a DataFrame and save to CSV
results_df = pd.DataFrame(results)
results_df.to_csv('gaussian_blur_results.csv', index=False)

# Plotting results
plt.figure(figsize=(12, 6))
plt.bar([r['Variant'] for r in results], [r['PSNR'] for r in results])
plt.title('PSNR Results for Gaussian Blur')
plt.xlabel('Image Variants')
plt.ylabel('PSNR')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('gaussian_blur_psnr_results.png')
plt.show()


In [ ]:
output_folder = 'data_enhanced/denoised'  # Output folder for enhanced images

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

results = []

for filename in os.listdir(input_folder):
    if filename.endswith('.jpg') or filename.endswith('.png'):
        img_path = os.path.join(input_folder, filename)
        original = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Gaussian Blur (Denoising)
        enhanced = cv2.GaussianBlur(original, (5, 5), 0)
        psnr = calculate_psnr(original, enhanced)

        # Save enhanced image
        cv2.imwrite(os.path.join(output_folder, filename), enhanced)

        # Store results
        results.append({
            'Technique': 'Denoising',
            'Variant': filename,
            'PSNR': psnr
        })

results_df = pd.DataFrame(results)
results_df.to_csv('denoised_results.csv', index=False)

# Plotting results
plt.figure(figsize=(12, 6))
plt.bar([r['Variant'] for r in results], [r['PSNR'] for r in results])
plt.title('PSNR Results for Denoising')
plt.xlabel('Image Variants')
plt.ylabel('PSNR')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('denoised_psnr_results.png')
plt.show()


In [ ]:
output_folder = 'data_enhanced/unsharp'  # Output folder for enhanced images

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

results = []

for filename in os.listdir(input_folder):
    if filename.endswith('.jpg') or filename.endswith('.png'):
        img_path = os.path.join(input_folder, filename)
        original = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Unsharp Masking
        blurred = cv2.GaussianBlur(original, (5, 5), 0)
        enhanced = cv2.addWeighted(original, 1.5, blurred, -0.5, 0)
        psnr = calculate_psnr(original, enhanced)

        # Save enhanced image
        cv2.imwrite(os.path.join(output_folder, filename), enhanced)

        # Store results
        results.append({
            'Technique': 'Unsharp Masking',
            'Variant': filename,
            'PSNR': psnr
        })

results_df = pd.DataFrame(results)
results_df.to_csv('unsharp_results.csv', index=False)

# Plotting results
plt.figure(figsize=(12, 6))
plt.bar([r['Variant'] for r in results], [r['PSNR'] for r in results])
plt.title('PSNR Results for Unsharp Masking')
plt.xlabel('Image Variants')
plt.ylabel('PSNR')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('unsharp_psnr_results.png')
plt.show()


In [ ]:
output_folder = 'data_enhanced/retinex'  # Output folder for enhanced images

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

results = []

for filename in os.listdir(input_folder):
    if filename.endswith('.jpg') or filename.endswith('.png'):
        img_path = os.path.join(input_folder, filename)
        original = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Simple Retinex Enhancement
        enhanced = cv2.xphoto.createSimpleWB().balanceWhite(original)
        psnr = calculate_psnr(original, enhanced)

        # Save enhanced image
        cv2.imwrite(os.path.join(output_folder, filename), enhanced)

        # Store results
        results.append({
            'Technique': 'Retinex Enhancement',
            'Variant': filename,
            'PSNR': psnr
        })

results_df = pd.DataFrame(results)
results_df.to_csv('retinex_results.csv', index=False)

# Plotting results
plt.figure(figsize=(12, 6))
plt.bar([r['Variant'] for r in results], [r['PSNR'] for r in results])
plt.title('PSNR Results for Retinex Enhancement')
plt.xlabel('Image Variants')
plt.ylabel('PSNR')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('retinex_psnr_results.png')
plt.show()


In [ ]:
import cv2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def calculate_psnr(original, enhanced):
    mse = np.mean((original - enhanced) ** 2)
    if mse == 0:
        return float('inf')
    psnr = 20 * np.log10(255.0 / np.sqrt(mse))
    return psnr

# Folder path to images
input_folder = 'data/fog'  # Change to your fog class images path
output_folder = 'data_enhanced/clahe'  # Output folder for enhanced images

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

results = []

for filename in os.listdir(input_folder):
    if filename.endswith('.jpg') or filename.endswith('.png'):
        img_path = os.path.join(input_folder, filename)
        original = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # CLAHE Enhancement
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(original)
        psnr = calculate_psnr(original, enhanced)

        # Save enhanced image
        cv2.imwrite(os.path.join(output_folder, filename), enhanced)

        # Store results
        results.append({
            'Technique': 'CLAHE',
            'Variant': filename,
            'PSNR': psnr
        })

# Create a DataFrame and save to CSV
results_df = pd.DataFrame(results)
results_df.to_csv('clahe_results.csv', index=False)

# Plotting results
plt.figure(figsize=(12, 6))
plt.bar([r['Variant'] for r in results], [r['PSNR'] for r in results])
plt.title('PSNR Results for CLAHE')
plt.xlabel('Image Variants')
plt.ylabel('PSNR')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('clahe_psnr_results.png')
plt.show()


In [ ]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt

# Function to show images before and after enhancement
def display_images(original, enhanced, title):
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.title('Original Image')
    plt.imshow(original, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title(title)
    plt.imshow(enhanced, cmap='gray')
    plt.axis('off')

    plt.show()

# Folder path to images
input_folder = 'data/fog'  # Change to your fog class images path
output_folder = 'data_enhanced/denoising'  # Output folder for denoised images

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

for filename in os.listdir(input_folder):
    if filename.endswith('.jpg') or filename.endswith('.png'):
        img_path = os.path.join(input_folder, filename)
        original = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Denoising using Non-Local Means
        denoised = cv2.fastNlMeansDenoising(original, None, h=10, templateWindowSize=7, searchWindowSize=21)

        # Save denoised image
        cv2.imwrite(os.path.join(output_folder, filename), denoised)

        # Display images
        display_images(original, denoised, 'Denoised Image')


In [ ]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt

# Function to show images before and after enhancement
def display_images(original, enhanced, title):
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.title('Original Image')
    plt.imshow(original, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title(title)
    plt.imshow(enhanced, cmap='gray')
    plt.axis('off')

    plt.show()

# Folder path to images
input_folder = 'data/fog'  # Change to your fog class images path
output_folder = 'data_enhanced/unsharp'  # Output folder for unsharp images

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

for filename in os.listdir(input_folder):
    if filename.endswith('.jpg') or filename.endswith('.png'):
        img_path = os.path.join(input_folder, filename)
        original = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Unsharp Masking
        gaussian_blur = cv2.GaussianBlur(original, (5, 5), 1.5)
        unsharp_masked = cv2.addWeighted(original, 1.5, gaussian_blur, -0.5, 0)

        # Save unsharp image
        cv2.imwrite(os.path.join(output_folder, filename), unsharp_masked)

        # Display images
        display_images(original, unsharp_masked, 'Unsharp Masked Image')


In [ ]:
import cv2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def calculate_psnr(original, enhanced):
    mse = np.mean((original - enhanced) ** 2)
    if mse == 0:
        return float('inf')  # Infinite PSNR if no error
    psnr = 20 * np.log10(255.0 / np.sqrt(mse))
    return psnr

# Function to display images before and after enhancement
def display_images(original, enhanced, title):
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.title('Original Image')
    plt.imshow(original, cmap='gray')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title(title)
    plt.imshow(enhanced, cmap='gray')
    plt.axis('off')

    plt.show()

# Folder path to images
input_folder = 'data/fog'  # Change to your fog class images path
output_folder = 'data_enhanced/lmar'  # Output folder for LMAR-like images

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

results = []  # To store PSNR results

for filename in os.listdir(input_folder):
    if filename.endswith('.jpg') or filename.endswith('.png'):
        img_path = os.path.join(input_folder, filename)
        original = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Downsampling the image
        downsampled = cv2.resize(original, (original.shape[1] // 2, original.shape[0] // 2), interpolation=cv2.INTER_LINEAR)

        # Apply enhancement (sharpen filter) to the downsampled image
        gaussian_blur = cv2.GaussianBlur(downsampled, (5, 5), 1.5)
        enhanced_downsampled = cv2.addWeighted(downsampled, 1.5, gaussian_blur, -0.5, 0)

        # Upsample back to original resolution
        upsampled = cv2.resize(enhanced_downsampled, (original.shape[1], original.shape[0]), interpolation=cv2.INTER_LINEAR)

        # Compensatory Feature Injection
        compensatory_image = cv2.addWeighted(upsampled, 0.7, original, 0.3, 0)  # Example of blending

        # Calculate PSNR for enhanced image
        psnr = calculate_psnr(original, compensatory_image)

        # Save enhanced image
        cv2.imwrite(os.path.join(output_folder, filename), compensatory_image)

        # Store results
        results.append({
            'Technique': 'LMAR-like',
            'Variant': filename,
            'PSNR': psnr
        })

# Create a DataFrame and save to CSV
results_df = pd.DataFrame(results)
results_df.to_csv('lmar_results.csv', index=False)

# Plotting PSNR results
plt.figure(figsize=(12, 6))
plt.bar([r['Variant'] for r in results], [r['PSNR'] for r in results], color='orange')
plt.title('PSNR Results for LMAR-like Enhancement')
plt.xlabel('Image Variants')
plt.ylabel('PSNR (dB)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('lmar_psnr_results.png')
plt.show()


In [ ]:
import cv2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Function to calculate PSNR
def calculate_psnr(original, enhanced):
    mse = np.mean((original - enhanced) ** 2)
    if mse == 0:
        return float('inf')
    psnr = 20 * np.log10(255.0 / np.sqrt(mse))
    return psnr

# Define different techniques for enhancement
techniques = ['GaussianBlur', 'Denoising', 'Unsharp Masking', 'LMAR', 'CLAHE']

# Folder paths for input images
input_folder = 'data/fog'  # Change this as necessary

# Initialize results dictionary
results = {technique: [] for technique in techniques}

# Enhancement and PSNR calculation for each technique
for filename in os.listdir(input_folder):
    if filename.endswith('.jpg') or filename.endswith('.png'):
        img_path = os.path.join(input_folder, filename)
        original = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Apply Gaussian Blur
        gaussian_blur = cv2.GaussianBlur(original, (5, 5), 1.5)
        results['GaussianBlur'].append(calculate_psnr(original, gaussian_blur))

        # Apply Denoising
        denoised = cv2.fastNlMeansDenoising(original, None, 30, 7, 21)
        results['Denoising'].append(calculate_psnr(original, denoised))

        # Apply Unsharp Masking
        blur = cv2.GaussianBlur(original, (5, 5), 1.5)
        unsharp_masked = cv2.addWeighted(original, 1.5, blur, -0.5, 0)
        results['Unsharp Masking'].append(calculate_psnr(original, unsharp_masked))

        # LMAR-like method
        downsampled = cv2.resize(original, (original.shape[1] // 2, original.shape[0] // 2))
        gaussian_blur = cv2.GaussianBlur(downsampled, (5, 5), 1.5)
        enhanced_downsampled = cv2.addWeighted(downsampled, 1.5, gaussian_blur, -0.5, 0)
        upsampled = cv2.resize(enhanced_downsampled, (original.shape[1], original.shape[0]))
        lmar = cv2.addWeighted(upsampled, 0.7, original, 0.3, 0)
        results['LMAR'].append(calculate_psnr(original, lmar))

        # Apply CLAHE
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(original)
        results['CLAHE'].append(calculate_psnr(original, enhanced))

# Calculate the average PSNR for each technique
avg_psnr = {technique: np.mean(results[technique]) for technique in techniques}

# Create a DataFrame for easier plotting
psnr_df = pd.DataFrame(list(avg_psnr.items()), columns=['Technique', 'Average PSNR'])

# Plotting the bar chart
plt.figure(figsize=(10, 6))
plt.bar(psnr_df['Technique'], psnr_df['Average PSNR'], color=['blue', 'orange', 'pink', 'red', 'purple'])
plt.title('Comparison of PSNR Across Image Enhancement Techniques')
plt.xlabel('Enhancement Technique')
plt.ylabel('Average PSNR')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('psnr_comparison_bar_chart.png')
plt.show()


In [ ]:
import cv2
import os
import numpy as np

# Function to apply Gaussian blur
def apply_gaussian_blur(image):
    return cv2.GaussianBlur(image, (5, 5), 1.5)

# Function to apply CLAHE
def apply_clahe(image):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(image)

# Function to apply Unsharp Masking
def apply_unsharp_mask(image):
    gaussian = cv2.GaussianBlur(image, (9, 9), 10.0)
    return cv2.addWeighted(image, 1.5, gaussian, -0.5, 0)

# Function to apply denoising
def apply_denoising(image):
    return cv2.fastNlMeansDenoising(image, None, 10, 7, 21)

# Function to apply LMAR (placeholder for LMAR-like enhancement)
def apply_lmar(image):
    downsampled = cv2.resize(image, (image.shape[1] // 2, image.shape[0] // 2))
    gaussian_blur = cv2.GaussianBlur(downsampled, (5, 5), 1.5)
    enhanced_downsampled = cv2.addWeighted(downsampled, 1.5, gaussian_blur, -0.5, 0)
    upsampled = cv2.resize(enhanced_downsampled, (image.shape[1], image.shape[0]))
    return cv2.addWeighted(upsampled, 0.7, image, 0.3)

# Apply all techniques to the images in a folder
def enhance_images(input_folder, output_folder, technique):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    for filename in os.listdir(input_folder):
        if filename.endswith('.jpg') or filename.endswith('.png'):
            img_path = os.path.join(input_folder, filename)
            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if technique == "Gaussian Blur":
                enhanced = apply_gaussian_blur(image)
            elif technique == "CLAHE":
                enhanced = apply_clahe(image)
            elif technique == "Unsharp Masking":
                enhanced = apply_unsharp_mask(image)
            elif technique == "Denoising":
                enhanced = apply_denoising(image)
            elif technique == "LMAR":
                enhanced = apply_lmar(image)

            cv2.imwrite(os.path.join(output_folder, filename), enhanced)

# Paths to your input and output folders
input_folder = 'data/fog'  # Example folder
output_folders = {
    "Gaussian Blur": "data_enhanced/gaussian_blur",
    "CLAHE": "data_enhanced/clahe",
    "Unsharp Masking": "data_enhanced/unsharp_masking",
    "Denoising": "data_enhanced/denoising",
    "LMAR": "data_enhanced/lmar"
}

# Apply all techniques
for technique, output_folder in output_folders.items():
    enhance_images(input_folder, output_folder, technique)
